# 🏗️ Multi-Model Architecture Notebook — Drone Detection
**Separate from benchmarking.**  This notebook defines, builds, and trains each model
using the *best architecture* for the drone detection task.

| # | Model | Best Architecture | Strength |
|---|-------|------------------|---------|
| 1 | **YOLOv8s** | Ultralytics native | Fastest; best for real-time |
| 2 | **RT-DETR-L** | Ultralytics native | Highest accuracy; transformer |
| 3 | **Faster R-CNN** | ResNet-50 FPN v2 + FPN | Best 2-stage accuracy |
| 4 | **SSD** | SSDLite MobileNetV3-Large | Best speed/size for edge |

> **Classes:** `0: AirPlane` · `1: Drone` · `2: Helicopter`  
> ⚙️ Run each section independently — comment out any model block to skip.


## 1 · Imports & Install

In [ ]:
# Uncomment to install if needed:
# !pip install -q ultralytics torchvision torch tqdm albumentations

import os, time, random
import numpy as np
import cv2
import torch
import torchvision
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from glob import glob
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

# ── Torchvision detection imports ─────────────────────────────────────────────
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,           # BEST Faster-RCNN arch
    ssdlite320_mobilenet_v3_large,        # BEST SSD arch
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.ops import box_iou

from ultralytics import YOLO

print(f"PyTorch {torch.__version__}  |  Torchvision {torchvision.__version__}")


## 2 · Global Configuration  ← edit here

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
DATASET_ROOT = "./drone-dataset"
DATA_YAML    = os.path.join(DATASET_ROOT, "data.yaml")

# ── Classes ───────────────────────────────────────────────────────────────────
CLASS_NAMES = ["AirPlane", "Drone", "Helicopter"]
NUM_CLASSES = len(CLASS_NAMES)          # 3  (add +1 for BG inside each loader)

# ── Training hypers (shared defaults — override per-model below) ───────────────
EPOCHS       = 30
BATCH_SIZE   = 8
IMG_SIZE     = 640             # YOLO / RT-DETR
LEARNING_RATE= 0.005
WEIGHT_DECAY = 5e-4
MOMENTUM     = 0.9
CONF_THRESH  = 0.30
IOU_THRESH   = 0.50            # for NMS / mAP

# ── Toggle which model to train/evaluate ──────────────────────────────────────
# Set False to skip that model's section
TRAIN_YOLO   = False   # usually trained via Ultralytics CLI — see section 3
TRAIN_RTDETR = False   # same
TRAIN_FRCNN  = True    # ResNet-50-FPN-v2
TRAIN_SSD    = True    # SSDLite-MNv3-Large

# ── Paths for saving weights ───────────────────────────────────────────────────
FRCNN_SAVE = "fasterrcnn_resnet50_fpnv2_drone.pth"
SSD_SAVE   = "ssd_mobilenetv3_drone.pth"

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps");  print("🍎 Apple MPS")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda"); print(f"⚡ CUDA: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device("cpu");  print("🖥️  CPU")


## 3 · Shared Dataset Class (YOLO-format labels)

Used by Faster R-CNN and SSD.  
YOLO / RT-DETR consume the `data.yaml` directly via Ultralytics.


In [ ]:
class DroneDataset(Dataset):
    """
    Loads images + YOLO-format labels for Torchvision models.

    Label format per line:  class_id  cx  cy  bw  bh  (all normalised 0-1)
    Returns image tensor [C,H,W] ∈ [0,1] and target dict with:
        boxes  : FloatTensor (N, 4)  in xyxy pixel coords
        labels : Int64Tensor (N,)    1-indexed (0 = background)
    """
    def __init__(self, root, split="train", img_size=None):
        self.img_dir  = os.path.join(root, split, "images")
        self.lbl_dir  = os.path.join(root, split, "labels")
        self.img_size = img_size   # optional resize (H, W)
        self.imgs = sorted([f for f in os.listdir(self.img_dir)
                            if f.lower().endswith((".jpg",".png"))])
        print(f"  📂 {split:5s}: {len(self.imgs)} images ({self.img_dir})")

    # ────────────────────────────────────────────────────────────────────────
    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        fname    = self.imgs[idx]
        img_path = os.path.join(self.img_dir, fname)
        lbl_path = os.path.join(self.lbl_dir, os.path.splitext(fname)[0] + ".txt")

        # ── Load & normalise image ─────────────────────────────────────────
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        if self.img_size:
            img = cv2.resize(img, (self.img_size[1], self.img_size[0]))
        H, W = img.shape[:2]
        img_t = torch.from_numpy(img.astype(np.float32) / 255.0).permute(2,0,1)

        # ── Parse YOLO labels → xyxy ───────────────────────────────────────
        boxes, labels = [], []
        if os.path.exists(lbl_path):
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5: continue
                    c, cx, cy, bw, bh = int(parts[0]), *map(float, parts[1:5])
                    x1 = (cx - bw/2) * W;  y1 = (cy - bh/2) * H
                    x2 = (cx + bw/2) * W;  y2 = (cy + bh/2) * H
                    # clamp to image bounds
                    x1,y1 = max(0.,x1), max(0.,y1)
                    x2,y2 = min(float(W),x2), min(float(H),y2)
                    if x2 > x1 and y2 > y1:
                        boxes.append([x1,y1,x2,y2])
                        labels.append(c + 1)          # 0 = BG; classes start at 1

        target = {
            "boxes" : torch.as_tensor(boxes,  dtype=torch.float32) if boxes else torch.zeros((0,4)),
            "labels": torch.as_tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64),
            "image_id": torch.tensor([idx]),
        }
        return img_t, target

def collate_fn(batch): return tuple(zip(*batch))

print("✅ DroneDataset defined")


## 4 · YOLOv8s — Best Architecture for Drone Detection

**Why YOLOv8s?**
- **C2f backbone** with improved gradient flow vs YOLOv5
- **Anchor-free head** — faster training convergence
- Built-in **mosaic + mixup** augmentation for small targets
- `s` (small) variant: optimal speed-accuracy trade-off for aerial imagery at 640 px

> **Training is done via the Ultralytics API** (not a manual loop).  
> Simply run the cell below. Weights are saved to `runs/detect/yolov8s_drone/`.


In [ ]:
if TRAIN_YOLO:
    # ── YOLOv8s — Best Architecture ───────────────────────────────────────────
    # Starts from COCO pretrained weights (`yolov8s.pt`) and fine-tunes on your
    # 3-class drone dataset.  For best results:
    #   · imgsz=640 (matches your dataset)
    #   · augment mosaic/mixup enabled by default
    #   · amp=True  (mixed precision — faster on GPU)

    model = YOLO("yolov8s.pt")    # pretrained COCO checkpoint

    results = model.train(
        data      = DATA_YAML,
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH_SIZE,
        device    = 0 if torch.cuda.is_available() else "cpu",
        project   = "runs/detect",
        name      = "yolov8s_drone",
        exist_ok  = True,
        amp       = True,          # mixed precision
        patience  = 10,            # early stopping
        # ── augmentation (already on by default; tune if needed) ─────────────
        degrees   = 5.0,           # rotation ±5°
        translate = 0.1,
        scale     = 0.5,
        fliplr    = 0.5,
        mosaic    = 1.0,
        mixup     = 0.15,
    )
    print("\n✅ YOLOv8s training complete")
    print("  Best weights → runs/detect/yolov8s_drone/weights/best.pt")
else:
    print("⏭️  YOLO training skipped  (TRAIN_YOLO = False)")


## 5 · RT-DETR-L — Best Transformer Architecture

**Why RT-DETR-L?**
- **Real-Time DEtection TRansformer** — no NMS post-processing needed
- HybridEncoder: CNN feature extraction + efficient transformer encoder
- Highest mAP of the 4 models, especially for occluded or distant objects
- `L` (large) variant: best accuracy; use `rtdetr-m.pt` if memory-constrained


In [ ]:
if TRAIN_RTDETR:
    from ultralytics import RTDETR

    # ── RT-DETR-L — Best Architecture ─────────────────────────────────────────
    # Pretrained on COCO; fine-tuned on 3-class drone dataset.
    # Key advantages over YOLO:
    #   · No NMS      → deterministic, faster post-processing
    #   · Transformer encoder captures global context → better for occluded drones

    rt_model = RTDETR("rtdetr-l.pt")    # COCO pretrained RT-DETR Large

    results = rt_model.train(
        data      = DATA_YAML,
        epochs    = EPOCHS,
        imgsz     = IMG_SIZE,
        batch     = BATCH_SIZE,
        device    = 0 if torch.cuda.is_available() else "cpu",
        project   = "runs/detect",
        name      = "rtdetr_l_drone",
        exist_ok  = True,
        amp       = True,
        patience  = 10,
    )
    print("\n✅ RT-DETR-L training complete")
    print("  Best weights → runs/detect/rtdetr_l_drone/weights/best.pt")
else:
    print("⏭️  RT-DETR training skipped  (TRAIN_RTDETR = False)")


## 6 · Faster R-CNN — Best Architecture: ResNet-50-FPN-v2

**Why ResNet-50-FPN-v2?**
- **ResNet-50 backbone** — far stronger than MobileNetV3 for accuracy
- **FPN v2** — improved feature pyramid (batch norm, extra conv)  → better small-object detection
- Pre-trained on **COCO** (`FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT`) — rich feature initialisation
- Best 2-stage accuracy for drone detection vs MobileNetV3 variant

> ⚠️ **Note:** This is a *different* architecture from `fasterrcnn_drone.pth`.
> Your existing `.pth` checkpoint was trained on MobileNetV3.
> Use this section if you want to retrain with the better backbone.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# Faster R-CNN — ResNet-50-FPN-v2  (BEST ARCHITECTURE)
# ═════════════════════════════════════════════════════════════════════════════

def build_fasterrcnn_best(num_classes):
    """
    Architecture:
      Backbone  : ResNet-50 (pretrained ImageNet via COCO weights)
      Neck      : Feature Pyramid Network v2 (improved BN + extra conv)
      RPN       : Region Proposal Network
      Head      : FastRCNNPredictor → (num_classes + 1)  [+1 = background]

    Improvement over MobileNetV3:
      · Deeper residual backbone → richer features for rare/small classes
      · FPN v2 → better multi-scale detection (drones at varying distances)
    """
    # Load COCO-pretrained backbone + FPN — replace the detection head only
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
    model   = fasterrcnn_resnet50_fpn_v2(weights=weights)

    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    return model


print("🏗️  Faster R-CNN (ResNet-50-FPN-v2) architecture:")
_tmp = build_fasterrcnn_best(NUM_CLASSES)
total_params = sum(p.numel() for p in _tmp.parameters())
trainable    = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable:,}")
del _tmp


In [ ]:
if TRAIN_FRCNN:
    # ── Data ──────────────────────────────────────────────────────────────────
    train_ds = DroneDataset(DATASET_ROOT, "train")
    val_ds   = DroneDataset(DATASET_ROOT, "valid")
    train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, collate_fn=collate_fn)
    val_dl   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, collate_fn=collate_fn)

    # ── Model ─────────────────────────────────────────────────────────────────
    frcnn = build_fasterrcnn_best(NUM_CLASSES).to(DEVICE)

    # ── Optimizer: SGD with warm cosine LR (standard for detection) ───────────
    params = [p for p in frcnn.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=LEARNING_RATE,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    lr_scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}

    print(f"\n🔥 Training Faster R-CNN (ResNet-50-FPN-v2) for {EPOCHS} epochs …")
    for epoch in range(1, EPOCHS + 1):

        # ── Train ─────────────────────────────────────────────────────────────
        frcnn.train()
        epoch_loss = 0.0
        for imgs, targets in tqdm(train_dl, desc=f"Ep {epoch:03d} train", leave=False):
            imgs    = [i.to(DEVICE) for i in imgs]
            targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
            loss_dict = frcnn(imgs, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item()

        avg_train = epoch_loss / len(train_dl)
        history["train_loss"].append(avg_train)

        # ── Validation (loss in train mode) ───────────────────────────────────
        val_loss = 0.0
        with torch.no_grad():
            frcnn.train()   # train mode gives loss dict
            for imgs, targets in tqdm(val_dl, desc=f"Ep {epoch:03d} val  ", leave=False):
                imgs    = [i.to(DEVICE) for i in imgs]
                targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
                val_loss += sum(frcnn(imgs, targets).values()).item()
        avg_val = val_loss / len(val_dl)
        history["val_loss"].append(avg_val)

        lr_scheduler.step()
        print(f"  Ep {epoch:03d}  train={avg_train:.4f}  val={avg_val:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.6f}")

        # ── Save best checkpoint ───────────────────────────────────────────────
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(frcnn.state_dict(), FRCNN_SAVE)
            print(f"         💾 Saved best → {FRCNN_SAVE}")

    print("\n✅ Faster R-CNN training complete")
else:
    print("⏭️  Faster R-CNN training skipped  (TRAIN_FRCNN = False)")


## 7 · SSD — Best Architecture: SSDLite MobileNetV3-Large 320

**Why SSDLite-MNv3-Large?**
- **Lightest** of the 4 models — ideal for real-time or edge/drone-board inference
- **Depthwise-separable convolutions** (SSDLite) vs standard SSD → ~3× fewer FLOPS
- **MobileNetV3-Large** backbone — best accuracy/speed ratio in the MobileNet family
- Anchor boxes pre-set at 320 × 320 — matches small object scales in aerial imagery

> ⚠️ The head channel list `[672, 480, 512, 256, 256, 128]` is **fixed by the backbone**
> and must not change.  Changing it will break weight loading.


In [ ]:
# ═════════════════════════════════════════════════════════════════════════════
# SSD — SSDLite MobileNetV3-Large 320  (BEST ARCHITECTURE)
# ═════════════════════════════════════════════════════════════════════════════

def build_ssd_best(num_classes):
    """
    Architecture:
      Backbone  : MobileNetV3-Large  (pretrained ImageNet)
      Detector  : SSDLite (depthwise-separable convs — 3× fewer FLOPs than SSD)
      Input     : 320 × 320 px
      Head in_channels: [672, 480, 512, 256, 256, 128]
                         ↑ fixed by MobileNetV3-Large feature maps — do NOT change
      Output    : (num_classes + 1) categories per anchor  [+1 = background]

    Best-practice: initialise from 'DEFAULT' COCO weights, replace head only.
    """
    model       = ssdlite320_mobilenet_v3_large(weights="DEFAULT")
    in_channels = [672, 480, 512, 256, 256, 128]   # MNv3-Large backbone channels
    num_anchors = model.anchor_generator.num_anchors_per_location()
    model.head.classification_head = SSDClassificationHead(
        in_channels, num_anchors, num_classes + 1
    )
    return model

print("🏗️  SSD (SSDLite-MNv3-Large-320) architecture:")
_tmp = build_ssd_best(NUM_CLASSES)
total_params = sum(p.numel() for p in _tmp.parameters())
trainable    = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"   Total params    : {total_params:,}")
print(f"   Trainable params: {trainable:,}")
del _tmp


In [ ]:
if TRAIN_SSD:
    # ── Data (SSD takes 320×320) ───────────────────────────────────────────────
    ssd_train_ds = DroneDataset(DATASET_ROOT, "train", img_size=(320,320))
    ssd_val_ds   = DroneDataset(DATASET_ROOT, "valid", img_size=(320,320))
    ssd_train_dl = DataLoader(ssd_train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=2, collate_fn=collate_fn)
    ssd_val_dl   = DataLoader(ssd_val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=2, collate_fn=collate_fn)

    # ── Model ─────────────────────────────────────────────────────────────────
    ssd = build_ssd_best(NUM_CLASSES).to(DEVICE)

    # ── Optimizer: SGD (standard for SSD) ─────────────────────────────────────
    params    = [p for p in ssd.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(params, lr=LEARNING_RATE,
                                momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
    lr_scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer, milestones=[int(EPOCHS*0.6), int(EPOCHS*0.8)], gamma=0.1
    )

    # ── Training loop ─────────────────────────────────────────────────────────
    best_val_loss = float("inf")
    ssd_history   = {"train_loss": [], "val_loss": []}

    print(f"\n🔥 Training SSD (SSDLite-MNv3-Large) for {EPOCHS} epochs …")

    for epoch in range(1, EPOCHS + 1):
        # Train
        ssd.train()
        ep_loss = 0.0
        for imgs, targets in tqdm(ssd_train_dl, desc=f"Ep {epoch:03d} train", leave=False):
            imgs    = [i.to(DEVICE) for i in imgs]
            targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
            loss_dict = ssd(imgs, targets)
            loss = sum(loss_dict.values())
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            ep_loss += loss.item()

        avg_train = ep_loss / len(ssd_train_dl)
        ssd_history["train_loss"].append(avg_train)

        # Validation
        val_loss = 0.0
        with torch.no_grad():
            ssd.train()
            for imgs, targets in tqdm(ssd_val_dl, desc=f"Ep {epoch:03d} val  ", leave=False):
                imgs    = [i.to(DEVICE) for i in imgs]
                targets = [{k: v.to(DEVICE) for k,v in t.items()} for t in targets]
                val_loss += sum(ssd(imgs, targets).values()).item()

        avg_val = val_loss / len(ssd_val_dl)
        ssd_history["val_loss"].append(avg_val)
        lr_scheduler.step()

        print(f"  Ep {epoch:03d}  train={avg_train:.4f}  val={avg_val:.4f}  "
              f"lr={optimizer.param_groups[0]['lr']:.6f}")

        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(ssd.state_dict(), SSD_SAVE)
            print(f"         💾 Saved best → {SSD_SAVE}")

    print("\n✅ SSD training complete")
else:
    print("⏭️  SSD training skipped  (TRAIN_SSD = False)")


## 8 · Training Loss Curves

In [ ]:
# Plot whichever models were actually trained
trained = {}
if TRAIN_FRCNN and "history" in dir():     trained["Faster R-CNN (ResNet50-FPN-v2)"] = history
if TRAIN_SSD   and "ssd_history" in dir(): trained["SSD (SSDLite-MNv3-Large)"]       = ssd_history

if not trained:
    print("No Torchvision models were trained in this run.")
else:
    cols   = len(trained)
    fig, axes = plt.subplots(1, cols, figsize=(6*cols, 4.5))
    if cols == 1: axes = [axes]

    for ax, (name, hist) in zip(axes, trained.items()):
        epochs = range(1, len(hist["train_loss"]) + 1)
        ax.plot(epochs, hist["train_loss"], label="Train", color="#4f86c6", linewidth=1.8)
        ax.plot(epochs, hist["val_loss"],   label="Val",   color="#e44d42", linewidth=1.8, ls="--")
        ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
        ax.set_title(name, fontweight="bold"); ax.legend()
        ax.spines[["top","right"]].set_visible(False)

    plt.suptitle("Training Loss Curves", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig("training_loss_curves.png", bbox_inches="tight")
    plt.show()
    print("💾 training_loss_curves.png")


## 9 · Architecture Comparison Table

In [ ]:
# ── Parameter counts for all 4 best architectures ────────────────────────────
import pandas as pd

rows = []

# YOLOv8s
try:
    m = YOLO("yolov8s.pt").model
    rows.append({"Model":"YOLOv8s",
                 "Backbone":"CSP-Darknet (C2f)","Neck":"FPN+PAN","Head":"Anchor-free Decoupled",
                 "Params (M)":round(sum(p.numel() for p in m.parameters())/1e6,1),
                 "Input":"640×640","Pretrained":"COCO",
                 "Notes":"Best speed/size; anchor-free"})
    del m
except Exception as e:
    rows.append({"Model":"YOLOv8s","Backbone":"CSP-Darknet (C2f)","Neck":"FPN+PAN","Head":"Anchor-free",
                 "Params (M)":"~11.1","Input":"640×640","Pretrained":"COCO","Notes":"Best speed"})

# RT-DETR-L
rows.append({"Model":"RT-DETR-L","Backbone":"ResNet-50 + Hybrid Encoder",
             "Neck":"Efficient Transformer","Head":"Bipartite Matching (no NMS)",
             "Params (M)":"~32","Input":"640×640","Pretrained":"COCO",
             "Notes":"Highest accuracy; transformer; no NMS"})

# Faster R-CNN ResNet-50-FPN-v2
m2 = build_fasterrcnn_best(NUM_CLASSES)
rows.append({"Model":"Faster R-CNN (ResNet50-FPN-v2)",
             "Backbone":"ResNet-50","Neck":"FPN v2","Head":"RPN + RoI-Align + FastRCNN",
             "Params (M)":round(sum(p.numel() for p in m2.parameters())/1e6,1),
             "Input":"Variable","Pretrained":"COCO",
             "Notes":"Best 2-stage accuracy; strong for small objects"})
del m2

# SSD MNv3-Large
m3 = build_ssd_best(NUM_CLASSES)
rows.append({"Model":"SSD (SSDLite-MNv3-Large)",
             "Backbone":"MobileNetV3-Large","Neck":"SSDLite extras","Head":"SSD Classification",
             "Params (M)":round(sum(p.numel() for p in m3.parameters())/1e6,1),
             "Input":"320×320","Pretrained":"COCO",
             "Notes":"Fastest; best for edge/real-time"})
del m3

df_arch = pd.DataFrame(rows).set_index("Model")
print("\n📊 Best Architecture Comparison:\n")
display(df_arch.style
    .set_caption("Best Architectures for Drone Detection")
    .set_table_styles([
        {"selector":"caption","props":[("font-size","1.1em"),("font-weight","bold")]},
        {"selector":"th","props":[("background","#1e293b"),("color","white"),("padding","6px 10px")]},
        {"selector":"td","props":[("padding","5px 10px")]},
    ])
)


## 10 · Quick Inference Demo (trained models)

In [ ]:
# ── Load a saved checkpoint and run inference on a random test image ──────────

TEST_IMG_DIR = os.path.join(DATASET_ROOT, "test/images")
test_imgs    = sorted(glob(os.path.join(TEST_IMG_DIR, "*.jpg")))
sample_img   = random.choice(test_imgs)
print(f"Sample: {os.path.basename(sample_img)}")

def draw_boxes(img_bgr, boxes, scores, labels, class_names, conf=0.30):
    COLORS_CV = [(255,100,50),(50,200,100),(100,100,255),(255,200,0),(200,50,255)]
    out = img_bgr.copy()
    for box, score, lbl in zip(boxes, scores, labels):
        if score < conf: continue
        x1,y1,x2,y2 = map(int, box)
        cls_name = class_names[lbl] if 0 <= lbl < len(class_names) else f"cls{lbl}"
        c = COLORS_CV[lbl % len(COLORS_CV)]
        cv2.rectangle(out,(x1,y1),(x2,y2),c,2)
        label_txt = f"{cls_name} {score:.2f}"
        (tw,th),_ = cv2.getTextSize(label_txt, cv2.FONT_HERSHEY_SIMPLEX, 0.55, 1)
        cv2.rectangle(out,(x1,y1-th-6),(x1+tw+4,y1),c,-1)
        cv2.putText(out,label_txt,(x1+2,y1-4),cv2.FONT_HERSHEY_SIMPLEX,0.55,(255,255,255),1)
    return out

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
img_bgr   = cv2.imread(sample_img)
img_rgb   = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
axes[0].imshow(img_rgb); axes[0].set_title("Original"); axes[0].axis("off")

result_img = img_rgb.copy()   # replace with model output below

# ── Faster R-CNN inference (if weights exist) ──────────────────────────────
if os.path.exists(FRCNN_SAVE):
    frcnn_infer = build_fasterrcnn_best(NUM_CLASSES).to(DEVICE)
    frcnn_infer.load_state_dict(torch.load(FRCNN_SAVE, map_location=DEVICE))
    frcnn_infer.eval()
    t = torch.from_numpy(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
                         ).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = frcnn_infer(t)[0]
    boxes  = pred["boxes"].cpu().numpy()
    scores = pred["scores"].cpu().numpy()
    labels = pred["labels"].cpu().numpy().astype(int) - 1
    result_img = cv2.cvtColor(
        draw_boxes(img_bgr, boxes, scores, labels, CLASS_NAMES, CONF_THRESH),
        cv2.COLOR_BGR2RGB)
    axes[1].set_title(f"Faster R-CNN (ResNet50-FPN-v2)")
    del frcnn_infer
elif os.path.exists(SSD_SAVE):
    ssd_infer = build_ssd_best(NUM_CLASSES).to(DEVICE)
    ssd_infer.load_state_dict(torch.load(SSD_SAVE, map_location=DEVICE))
    ssd_infer.eval()
    img_320 = cv2.resize(img_bgr,(320,320))
    t = torch.from_numpy(cv2.cvtColor(img_320,cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
                         ).permute(2,0,1).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        pred = ssd_infer(t)[0]
    h0,w0 = img_bgr.shape[:2]
    boxes  = pred["boxes"].cpu().numpy() * np.array([w0/320,h0/320,w0/320,h0/320])
    scores = pred["scores"].cpu().numpy()
    labels = pred["labels"].cpu().numpy().astype(int) - 1
    result_img = cv2.cvtColor(
        draw_boxes(img_bgr, boxes, scores, labels, CLASS_NAMES, CONF_THRESH),
        cv2.COLOR_BGR2RGB)
    axes[1].set_title("SSD (SSDLite-MNv3-Large)")
    del ssd_infer
else:
    axes[1].set_title("No saved weights found — train first")

axes[1].imshow(result_img); axes[1].axis("off")
plt.suptitle("Inference Demo — Best Architecture", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.show()
